*Libraries And Build Dependencies*

In [1]:
import os
import time
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import spacy
import nltk
import warnings

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords as nltk_sw
nlp = spacy.load('en_core_web_md')
warnings.filterwarnings('ignore')

stage_times = {}
pipeline_start_time = time.perf_counter()

print(os.getcwd())
print('[Stage 0/4] Initialization complete.')

c:\Users\Dell\Desktop\Semester 6\Parallel & Distributed Computing\Project\PDC milestone 1
[Stage 0/4] Initialization complete.


In [2]:
if 'stage_times' not in globals():
    stage_times = {}

stage_start = time.perf_counter()
print('[Stage 1/4] Loading input data...')

# Load the CSV file
df = pd.read_csv('C:\\Users\\Dell\\Desktop\\Semester 6\\Parallel & Distributed Computing\\Project\\Final Datasets\\AlgoTesting_Merged_Coloums_Dataset.csv')

# Extract 'Combined_Data' and drop missing values
document = df['Combined_Data'].dropna().tolist()

elapsed = time.perf_counter() - stage_start
stage_times['data_loading_seconds'] = elapsed

print(f'Loaded {len(document)} documents.')
print(f'[Stage 1/4] Completed in {elapsed:.4f} seconds.')

[Stage 1/4] Loading input data...
Loaded 10253 documents.
[Stage 1/4] Completed in 0.3184 seconds.


**DATA PREPROCESSING**

We will make two set of data:
1. Without stopword removal and lemmatization for contextual embeddings
2. With stopword removal and lemmatization for TF-IDF

In [3]:
if 'stage_times' not in globals():
    stage_times = {}

stage_start = time.perf_counter()
print('[Stage 2/4] Preprocessing docs for TF-IDF (stopword removal + lemmatization)...')

#Defining stopwords list (Common words to remove from the text as they do not add much meaning to the analysis)
stopwords = set(nltk_sw.words('english'))

# Preprocessing the documents: Tokenization, Lemmatization, and Stopword Removal
no_stopword_docs= []
for d in document:
    d_nlp = nlp(d.lower())
    t_list = []
    for token in d_nlp:
        tok_lem = str(token.lemma_)
        if (tok_lem not in stopwords) and (tok_lem.isalpha()):
            t_list.append(tok_lem)
    str_ = ' '.join(t_list) 
    no_stopword_docs.append(str_)

elapsed = time.perf_counter() - stage_start
stage_times['preprocessing_tfidf_seconds'] = elapsed
print(f'[Stage 2/4] TF-IDF preprocessing completed in {elapsed:.4f} seconds.')

no_stopword_docs

[Stage 2/4] Preprocessing docs for TF-IDF (stopword removal + lemmatization)...
[Stage 2/4] TF-IDF preprocessing completed in 854.8450 seconds.


['emergency snf marry white unspecified protein calorie malnutrition cellulitis abscess leg except foot congestive heart failure unspecified subendocardial infarction initial episode care primary cardiomyopathy acute kidney failure unspecified shock without mention trauma unspecified septicemia cardiac arrest yeast gram negative probable enterococcus staphylococcus coagulase negative',
 'emergency home home iv providr single white antiviral drug cause adverse effect therapeutic use human immunodeficiency virus hiv disease pneumocystosis cachexia alkalosis bacteremia cirrhosis liver without mention alcohol methicillin susceptible staphylococcus aureus condition classify elsewhere unspecified site infection microorganism resistant penicillin mucolytic expectorant antidote anaerobic antibiotic iv fluid dextrose base thyroid hormone miscellaneous non opioid analgesic antipyretic laxative stool softener corticosteroid anti inflammatory laxative stool softener antagonist antifungal injectabl

In [4]:
if 'stage_times' not in globals():
    stage_times = {}

stage_start = time.perf_counter()
print('[Stage 2/4] Preparing raw docs for contextual embeddings...')

#Documents with only removal of symbols and lowercasing for contextual embeddings
#TODO: Remove the  "if token.text.isalpha():" as sentence seperation is important for contextual embeddings as they are trained on the original text and not on the preprocessed text.

raw_docs = []  # This list will hold the preprocessed documents without stopword removal
for d in document:
    d_nlp = nlp(d.lower())
    t_list = []
    for token in d_nlp:
        t_list.append(str(token))
    str_ = ' '.join(t_list)
    raw_docs.append(str_)

elapsed = time.perf_counter() - stage_start
stage_times['preprocessing_contextual_seconds'] = elapsed
stage_times['preprocessing_seconds'] = stage_times.get('preprocessing_tfidf_seconds', 0.0) + elapsed
print(f'[Stage 2/4] Contextual preprocessing completed in {elapsed:.4f} seconds.')
print(f'[Stage 2/4] Total preprocessing time: {stage_times["preprocessing_seconds"]:.4f} seconds.')

raw_docs

[Stage 2/4] Preparing raw docs for contextual embeddings...
[Stage 2/4] Contextual preprocessing completed in 867.1335 seconds.
[Stage 2/4] Total preprocessing time: 1721.9786 seconds.


['emergency | snf | married | white | unspecified protein - calorie malnutrition | cellulitis and abscess of leg , except foot | congestive heart failure , unspecified | subendocardial infarction , initial episode of care | other primary cardiomyopathies | acute kidney failure , unspecified | other shock without mention of trauma | unspecified septicemia | cardiac arrest | yeast | gram negative rod(s ) | probable enterococcus | staphylococcus , coagulase negative',
 'emergency | home with home iv providr | single | white | antiviral drugs causing adverse effects in therapeutic use | human immunodeficiency virus [ hiv ] disease | pneumocystosis | cachexia | alkalosis | bacteremia | cirrhosis of liver without mention of alcohol | methicillin susceptible staphylococcus aureus in conditions classified elsewhere and of unspecified site | infection with microorganisms resistant to penicillins | mucolytics / expectorants / antidotes | anaerobic antibiotics | iv fluids ( dextrose based ) | thy

**TF-IDF**

In [5]:
if 'stage_times' not in globals():
    stage_times = {}

stage_start = time.perf_counter()
print('[Stage 3/4] TF-IDF vectorization started...')

# Vectorization using CountVectorizer (Term Frequency)
count_vectorizer = CountVectorizer()
TF_IDF_Vector = count_vectorizer.fit_transform(no_stopword_docs).toarray()

elapsed = time.perf_counter() - stage_start
stage_times['tfidf_vectorization_seconds'] = elapsed
print(f'[Stage 3/4] Completed in {elapsed:.4f} seconds.')

TF_IDF_Vector.shape

[Stage 3/4] TF-IDF vectorization started...
[Stage 3/4] Completed in 2.1666 seconds.


(10253, 2923)

In [6]:
print("TF-IDF Vectorization completed. Shape of the vectorized data:", TF_IDF_Vector.shape)
print("Sample of the vectorized data (first 15 rows):")
print(TF_IDF_Vector[:15])

TF-IDF Vectorization completed. Shape of the vectorized data: (10253, 2923)
Sample of the vectorized data (first 15 rows):
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [7]:
#Joining the TF-IDF vector with original dataframe HADM_ID column for later use in similarity search
tfidf_df = pd.DataFrame(TF_IDF_Vector, columns=count_vectorizer.get_feature_names_out())
tfidf_df["HADM_ID"] = df['HADM_ID'][:len(tfidf_df)]

#Making the HADM_ID column the first column of the dataframe for better readability
cols = tfidf_df.columns.tolist()
cols = cols[-1:] + cols[:-1]
tfidf_df = tfidf_df[cols]


tfidf_df.head()



,HADM_ID,abdoman,abdominal,abducen,abiotrophia,abnormal,abnormality,abo,abortion,abrasion,...,worker,wound,wrist,wrong,xanthomonas,xi,yeast,youngae,zoster,zygomycosis
0,145834,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,185777,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,107064,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,150750,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,194540,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
if 'stage_times' not in globals():
    stage_times = {}

stage_start = time.perf_counter()

#saving the tfidf vector as csv file for later use in similarity search
tfidf_df.to_csv('C:\\Users\\Dell\\Desktop\\Semester 6\\Parallel & Distributed Computing\\Project\\Final Datasets\\tfidf_vector.csv', index=False)

save_elapsed = time.perf_counter() - stage_start
stage_times['output_saving_seconds'] = stage_times.get('output_saving_seconds', 0.0) + save_elapsed
print(f'TF-IDF CSV save completed in {save_elapsed:.4f} seconds.')

TF-IDF CSV save completed in 8.4384 seconds.


**Contextual Embeddings**

In [9]:
from sentence_transformers import SentenceTransformer
print("sentence-transformers imported OK")


sentence-transformers imported OK


In [10]:
if 'stage_times' not in globals():
    stage_times = {}

stage_start = time.perf_counter()
print('[Stage 4/4] BGE embedding started...')

print("Loading BAAI/bge-small-en-v1.5 ...")
bge_model      = SentenceTransformer('BAAI/bge-small-en-v1.5')
bge_embeddings = bge_model.encode(raw_docs, show_progress_bar=True,
                                  batch_size=32, convert_to_numpy=True)

elapsed = time.perf_counter() - stage_start
stage_times['bge_embedding_seconds'] = elapsed

print(f"BGE embedding shape: {bge_embeddings.shape}")
print(f"[Stage 4/4] Completed in {elapsed:.4f} seconds.")

[Stage 4/4] BGE embedding started...
Loading BAAI/bge-small-en-v1.5 ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 212.92it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 321/321 [40:43<00:00,  7.61s/it] 

BGE embedding shape: (10253, 384)
[Stage 4/4] Completed in 2451.6912 seconds.


In [11]:
print(bge_embeddings[:10])

[[-0.03283165 -0.02232037  0.0007639  ... -0.07025634  0.0609694
  -0.01485386]
 [-0.00806958 -0.02200694  0.00942245 ... -0.02727433  0.07534205
   0.01555719]
 [-0.03515428 -0.01281211  0.0500839  ... -0.04079188  0.04939627
  -0.00047593]
 ...
 [-0.05492511  0.01234658  0.0021915  ... -0.01995106  0.08971114
   0.04315965]
 [-0.03973247 -0.0126762   0.01461569 ... -0.02848863  0.09096858
   0.00757274]
 [-0.02995087  0.00513759  0.00836143 ... -0.07079654  0.03862844
   0.00354484]]


In [12]:
if 'stage_times' not in globals():
    stage_times = {}

stage_start = time.perf_counter()

#Joining the BGE embedding with original dataframe HADM_ID column for later use in similarity search
bge_df = pd.DataFrame(bge_embeddings)
bge_df['HADM_ID'] = df['HADM_ID'][:len(bge_df)]

#Making the HADM_ID column the first column of the dataframe for better readability
cols = bge_df.columns.tolist()
cols = cols[-1:] + cols[:-1]
bge_df = bge_df[cols]

print("BGE embedding DataFrame created successfully. Sample of the DataFrame:")
print(bge_df.head())

#saving the BGE embedding as csv file for later use in similarity search
bge_df.to_csv('C:\\Users\\Dell\\Desktop\\Semester 6\\Parallel & Distributed Computing\\Project\\Final Datasets\\bge_embedding.csv', index=False)

save_elapsed = time.perf_counter() - stage_start
stage_times['output_saving_seconds'] = stage_times.get('output_saving_seconds', 0.0) + save_elapsed
stage_times['total_pipeline_seconds'] = time.perf_counter() - globals().get('pipeline_start_time', time.perf_counter())

print(f'Output save step completed in {save_elapsed:.4f} seconds.')
print('\nTiming summary (seconds):')
for stage_name, stage_value in stage_times.items():
    print(f'  {stage_name}: {stage_value:.4f}')

BGE embedding DataFrame created successfully. Sample of the DataFrame:
   HADM_ID         0         1         2         3         4         5         6         7         8         9        10        11        12        13        14        15        16        17        18        19        20        21        22        23        24        25        26        27        28        29        30        31        32        33        34        35        36        37        38        39        40        41        42        43        44        45        46        47        48  ...       334       335       336       337       338       339       340       341       342       343       344       345       346       347       348       349       350       351       352       353       354       355       356       357       358       359       360       361       362       363       364       365       366       367       368       369       370       371       372       373       374       375    